In [1]:
import sys, json
from pathlib import Path
sys.path.insert(0, "../..")

import pandas as pd
import pyagrum as gum
import pyagrum.lib.notebook as gnb

from pipeline.Dataset import Dataset
from algorithms.CPCAdapter import CPCAdapter
from algorithms.CMIICAdapter import CMIICAdapter

VERSION = f"agrum{gum.__version__.split('.')[0]}"
RESULTS = Path("/tmp/agrum_compare")
(RESULTS / VERSION).mkdir(parents=True, exist_ok=True)
print(VERSION, "| pyagrum", gum.__version__)

/home/mathis/Documents/Travail/cbnsl_benchmark/venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


agrum2 | pyagrum 2.3.0


In [2]:
def make_dag(n_nodes, arcs):
    dag = gum.DAG()
    dag.addNodes(n_nodes)
    for tail, head in arcs:
        dag.addArc(tail, head)
    return dag

REPORT_DAG_10 = make_dag(10, [(1, 0), (0, 2), (2, 3), (3, 4), (3, 5), (4, 5), (4, 6),
                              (5, 7), (6, 7), (6, 8), (7, 8), (9, 8), (2, 9), (9, 7)])

TOY_PARAMS = {"CPC": {"alpha": 0.05, "max_conditioning_set_size": 3},
              "CMIIC": {"alpha": 0.05}}

REPORT_PARAMS = {"CPC": {"alpha": 0.1, "max_conditioning_set_size": 2},
                 "CMIIC": {"alpha": 0.01}}

GOLDENS = {
    "collider":     (make_dag(3, [(0, 2), (1, 2)]), TOY_PARAMS),
    "fork":         (make_dag(3, [(1, 0), (1, 2)]), TOY_PARAMS),
    "chain":        (make_dag(3, [(0, 1), (1, 2)]), TOY_PARAMS),
    "report_10v":   (REPORT_DAG_10, REPORT_PARAMS),
}

In [3]:
def save_dag(dag, path):
    content = {"n_nodes": dag.size(), "arcs": sorted([t, h] for t, h in dag.arcs())}
    path.write_text(json.dumps(content))

for name, (golden_dag, params) in GOLDENS.items():
    var_names = [f"X{i}" for i in range(golden_dag.size())]

    df = pd.read_csv(f"{name}_dataset.csv")
    dataset = Dataset(df.values, name=name, feature_names=var_names)

    cpc_dag = CPCAdapter(**params["CPC"]).learn_dag(dataset)
    cmiic_dag = CMIICAdapter(**params["CMIIC"]).learn_dag(dataset)

    save_dag(golden_dag, RESULTS / VERSION / f"{name}__golden.json")
    save_dag(cpc_dag, RESULTS / VERSION / f"{name}__CPC.json")
    save_dag(cmiic_dag, RESULTS / VERSION / f"{name}__CMIIC.json")

    print(name)
    print("  golden :", sorted(golden_dag.arcs()))
    print("  CPC    :", sorted(cpc_dag.arcs()))
    print("  CMIIC  :", sorted(cmiic_dag.arcs()))

collider
  golden : [(0, 2), (1, 2)]
  CPC    : [(0, 2), (1, 2)]
  CMIIC  : [(1, 2), (2, 0)]
fork
  golden : [(1, 0), (1, 2)]
  CPC    : [(1, 0), (2, 1)]
  CMIIC  : [(1, 0), (2, 1)]
chain
  golden : [(0, 1), (1, 2)]
  CPC    : [(1, 0), (2, 1)]
  CMIIC  : [(1, 0), (2, 1)]
report_10v
  golden : [(0, 2), (1, 0), (2, 3), (2, 9), (3, 4), (3, 5), (4, 5), (4, 6), (5, 7), (6, 7), (6, 8), (7, 8), (9, 7), (9, 8)]
  CPC    : [(0, 2), (1, 0), (3, 2), (3, 7), (3, 8), (4, 3), (4, 5), (4, 7), (4, 8), (5, 3), (5, 7), (5, 8), (6, 4), (6, 7), (6, 8), (7, 2), (8, 2), (8, 7), (9, 2), (9, 7), (9, 8)]
  CMIIC  : [(0, 1), (2, 0), (2, 3), (3, 4), (3, 5), (4, 5), (6, 4), (6, 8), (7, 5), (7, 6), (7, 8), (7, 9), (8, 9), (9, 2)]


In [4]:
from IPython.display import HTML, display

def dag_from_json(dag_json):
    dag = gum.DAG()
    dag.addNodes(dag_json["n_nodes"])
    for tail, head in dag_json["arcs"]:
        dag.addArc(tail, head)
    if hasattr(dag, "setName"):  # les DAG n'ont des noms que depuis aGrUM 3
        for i in range(dag.size()):
            dag.setName(i, f"X{i}")
    return dag

def bn_from_dag(dag):
    bn = gum.BayesNet()
    for i in range(dag.size()):
        bn.add(gum.LabelizedVariable(f"X{i}", "", 2))
    for tail, head in dag.arcs():
        bn.addArc(tail, head)
    return bn

def cpdag_of(dag):
    # garder bn et eg vivants pendant le calcul : EssentialGraph ne retient pas
    # le BayesNet cote Python, un temporaire serait libere trop tot (segfault)
    bn = bn_from_dag(dag)
    eg = gum.EssentialGraph(bn)
    return eg.pdag()

def skeleton_of(dag):
    bn = bn_from_dag(dag)
    eg = gum.EssentialGraph(bn)
    return eg.skeleton()

versions = sorted(p.name for p in RESULTS.iterdir() if p.is_dir())
print("versions :", versions)

CELL_STYLE = "text-align:center; vertical-align:middle; padding:8px"

for name in GOLDENS:
    dags = [dag_from_json(json.loads((RESULTS / versions[0] / f"{name}__golden.json").read_text()))]
    captions = [f"{name}<br>golden"]
    for algo in ["CPC", "CMIIC"]:
        for version in versions:
            learned_json = json.loads((RESULTS / version / f"{name}__{algo}.json").read_text())
            dags.append(dag_from_json(learned_json))
            captions.append(f"{algo}<br>{version}")

    header = "<th></th>" + "".join(f"<th style='{CELL_STYLE}'>{c}</th>" for c in captions)
    dag_cells = "".join(f"<td style='{CELL_STYLE}'>{gnb.getDot(d.toDot())}</td>" for d in dags)
    cpdag_cells = "".join(f"<td style='{CELL_STYLE}'>{gnb.getDot(cpdag_of(d).toDot())}</td>" for d in dags)
    skeleton_cells = "".join(f"<td style='{CELL_STYLE}'>{gnb.getDot(skeleton_of(d).toDot())}</td>" for d in dags)

    display(HTML(
        "<table>"
        f"<tr>{header}</tr>"
        f"<tr><th>DAG</th>{dag_cells}</tr>"
        f"<tr><th>CPDAG</th>{cpdag_cells}</tr>"
        f"<tr><th>Squelette</th>{skeleton_cells}</tr>"
        "</table>"
    ))

versions : ['agrum2', 'agrum3']


,collidergolden,CPCagrum2,CPCagrum3,CMIICagrum2,CMIICagrum3
DAG,G <!-- 0 --> 0 0 <!-- 2 --> 2 2 <!-- 0->2 --> 0->2 <!-- 1 --> 1 1 <!-- 1->2 --> 1->2,G <!-- 0 --> 0 0 <!-- 2 --> 2 2 <!-- 0->2 --> 0->2 <!-- 1 --> 1 1 <!-- 1->2 --> 1->2,G <!-- 0 --> 0 0 <!-- 2 --> 2 2 <!-- 0->2 --> 0->2 <!-- 1 --> 1 1 <!-- 1->2 --> 1->2,G <!-- 0 --> 0 0 <!-- 1 --> 1 1 <!-- 2 --> 2 2 <!-- 1->2 --> 1->2 <!-- 2->0 --> 2->0,G <!-- 0 --> 0 0 <!-- 1 --> 1 1 <!-- 2 --> 2 2 <!-- 1->2 --> 1->2 <!-- 2->0 --> 2->0
CPDAG,no_name <!-- 0 --> 0 0 <!-- 2 --> 2 2 <!-- 0->2 --> 0->2 <!-- 1 --> 1 1 <!-- 1->2 --> 1->2,no_name <!-- 0 --> 0 0 <!-- 2 --> 2 2 <!-- 0->2 --> 0->2 <!-- 1 --> 1 1 <!-- 1->2 --> 1->2,no_name <!-- 0 --> 0 0 <!-- 2 --> 2 2 <!-- 0->2 --> 0->2 <!-- 1 --> 1 1 <!-- 1->2 --> 1->2,no_name cluster_0 <!-- 1 --> 1 1 <!-- 2 --> 2 2 <!-- 1->2 --> 1->2 <!-- 0 --> 0 0 <!-- 0->2 --> 0->2,no_name cluster_0 <!-- 1 --> 1 1 <!-- 2 --> 2 2 <!-- 1->2 --> 1->2 <!-- 0 --> 0 0 <!-- 0->2 --> 0->2
Squelette,no_name <!-- 0 --> 0 0 <!-- 2 --> 2 2 <!-- 0->2 --> 0->2 <!-- 1 --> 1 1 <!-- 1->2 --> 1->2,no_name <!-- 0 --> 0 0 <!-- 2 --> 2 2 <!-- 0->2 --> 0->2 <!-- 1 --> 1 1 <!-- 1->2 --> 1->2,no_name <!-- 0 --> 0 0 <!-- 2 --> 2 2 <!-- 0->2 --> 0->2 <!-- 1 --> 1 1 <!-- 1->2 --> 1->2,no_name <!-- 0 --> 0 0 <!-- 2 --> 2 2 <!-- 0->2 --> 0->2 <!-- 1 --> 1 1 <!-- 1->2 --> 1->2,no_name <!-- 0 --> 0 0 <!-- 2 --> 2 2 <!-- 0->2 --> 0->2 <!-- 1 --> 1 1 <!-- 1->2 --> 1->2


,forkgolden,CPCagrum2,CPCagrum3,CMIICagrum2,CMIICagrum3
DAG,G <!-- 0 --> 0 0 <!-- 1 --> 1 1 <!-- 1->0 --> 1->0 <!-- 2 --> 2 2 <!-- 1->2 --> 1->2,G <!-- 0 --> 0 0 <!-- 1 --> 1 1 <!-- 1->0 --> 1->0 <!-- 2 --> 2 2 <!-- 2->1 --> 2->1,G <!-- 0 --> 0 0 <!-- 1 --> 1 1 <!-- 1->0 --> 1->0 <!-- 2 --> 2 2 <!-- 2->1 --> 2->1,G <!-- 0 --> 0 0 <!-- 1 --> 1 1 <!-- 1->0 --> 1->0 <!-- 2 --> 2 2 <!-- 2->1 --> 2->1,G <!-- 0 --> 0 0 <!-- 1 --> 1 1 <!-- 1->0 --> 1->0 <!-- 2 --> 2 2 <!-- 2->1 --> 2->1
CPDAG,no_name cluster_0 <!-- 1 --> 1 1 <!-- 2 --> 2 2 <!-- 1->2 --> 1->2 <!-- 0 --> 0 0 <!-- 0->1 --> 0->1,no_name cluster_0 <!-- 1 --> 1 1 <!-- 2 --> 2 2 <!-- 1->2 --> 1->2 <!-- 0 --> 0 0 <!-- 0->1 --> 0->1,no_name cluster_0 <!-- 1 --> 1 1 <!-- 2 --> 2 2 <!-- 1->2 --> 1->2 <!-- 0 --> 0 0 <!-- 0->1 --> 0->1,no_name cluster_0 <!-- 1 --> 1 1 <!-- 2 --> 2 2 <!-- 1->2 --> 1->2 <!-- 0 --> 0 0 <!-- 0->1 --> 0->1,no_name cluster_0 <!-- 1 --> 1 1 <!-- 2 --> 2 2 <!-- 1->2 --> 1->2 <!-- 0 --> 0 0 <!-- 0->1 --> 0->1
Squelette,no_name <!-- 0 --> 0 0 <!-- 1 --> 1 1 <!-- 0->1 --> 0->1 <!-- 2 --> 2 2 <!-- 1->2 --> 1->2,no_name <!-- 0 --> 0 0 <!-- 1 --> 1 1 <!-- 0->1 --> 0->1 <!-- 2 --> 2 2 <!-- 1->2 --> 1->2,no_name <!-- 0 --> 0 0 <!-- 1 --> 1 1 <!-- 0->1 --> 0->1 <!-- 2 --> 2 2 <!-- 1->2 --> 1->2,no_name <!-- 0 --> 0 0 <!-- 1 --> 1 1 <!-- 0->1 --> 0->1 <!-- 2 --> 2 2 <!-- 1->2 --> 1->2,no_name <!-- 0 --> 0 0 <!-- 1 --> 1 1 <!-- 0->1 --> 0->1 <!-- 2 --> 2 2 <!-- 1->2 --> 1->2


,chaingolden,CPCagrum2,CPCagrum3,CMIICagrum2,CMIICagrum3
DAG,G <!-- 0 --> 0 0 <!-- 1 --> 1 1 <!-- 0->1 --> 0->1 <!-- 2 --> 2 2 <!-- 1->2 --> 1->2,G <!-- 0 --> 0 0 <!-- 1 --> 1 1 <!-- 1->0 --> 1->0 <!-- 2 --> 2 2 <!-- 2->1 --> 2->1,G <!-- 0 --> 0 0 <!-- 1 --> 1 1 <!-- 1->0 --> 1->0 <!-- 2 --> 2 2 <!-- 2->1 --> 2->1,G <!-- 0 --> 0 0 <!-- 1 --> 1 1 <!-- 1->0 --> 1->0 <!-- 2 --> 2 2 <!-- 2->1 --> 2->1,G <!-- 0 --> 0 0 <!-- 1 --> 1 1 <!-- 1->0 --> 1->0 <!-- 2 --> 2 2 <!-- 2->1 --> 2->1
CPDAG,no_name cluster_0 <!-- 1 --> 1 1 <!-- 2 --> 2 2 <!-- 1->2 --> 1->2 <!-- 0 --> 0 0 <!-- 0->1 --> 0->1,no_name cluster_0 <!-- 1 --> 1 1 <!-- 2 --> 2 2 <!-- 1->2 --> 1->2 <!-- 0 --> 0 0 <!-- 0->1 --> 0->1,no_name cluster_0 <!-- 1 --> 1 1 <!-- 2 --> 2 2 <!-- 1->2 --> 1->2 <!-- 0 --> 0 0 <!-- 0->1 --> 0->1,no_name cluster_0 <!-- 1 --> 1 1 <!-- 2 --> 2 2 <!-- 1->2 --> 1->2 <!-- 0 --> 0 0 <!-- 0->1 --> 0->1,no_name cluster_0 <!-- 1 --> 1 1 <!-- 2 --> 2 2 <!-- 1->2 --> 1->2 <!-- 0 --> 0 0 <!-- 0->1 --> 0->1
Squelette,no_name <!-- 0 --> 0 0 <!-- 1 --> 1 1 <!-- 0->1 --> 0->1 <!-- 2 --> 2 2 <!-- 1->2 --> 1->2,no_name <!-- 0 --> 0 0 <!-- 1 --> 1 1 <!-- 0->1 --> 0->1 <!-- 2 --> 2 2 <!-- 1->2 --> 1->2,no_name <!-- 0 --> 0 0 <!-- 1 --> 1 1 <!-- 0->1 --> 0->1 <!-- 2 --> 2 2 <!-- 1->2 --> 1->2,no_name <!-- 0 --> 0 0 <!-- 1 --> 1 1 <!-- 0->1 --> 0->1 <!-- 2 --> 2 2 <!-- 1->2 --> 1->2,no_name <!-- 0 --> 0 0 <!-- 1 --> 1 1 <!-- 0->1 --> 0->1 <!-- 2 --> 2 2 <!-- 1->2 --> 1->2


In [5]:
assert len(versions) == 2, "il faut d'abord lancer dans les deux kernels (cbnsl et venv)"
first, second = versions

for name in GOLDENS:
    print(f"=== {name}")
    for algo in ["CPC", "CMIIC"]:
        arcs_first = json.loads((RESULTS / first / f"{name}__{algo}.json").read_text())["arcs"]
        arcs_second = json.loads((RESULTS / second / f"{name}__{algo}.json").read_text())["arcs"]
        if arcs_first == arcs_second:
            print(f"  {algo:6} identique")
        else:
            print(f"  {algo:6} different")
            print(f"    {first} : {arcs_first}")
            print(f"    {second} : {arcs_second}")
    print()

=== collider
  CPC    identique
  CMIIC  identique

=== fork
  CPC    identique
  CMIIC  identique

=== chain
  CPC    identique
  CMIIC  identique

=== report_10v
  CPC    identique
  CMIIC  different
    agrum2 : [[0, 1], [2, 0], [2, 3], [3, 4], [3, 5], [4, 5], [6, 4], [6, 8], [7, 5], [7, 6], [7, 8], [7, 9], [8, 9], [9, 2]]
    agrum3 : [[0, 1], [2, 0], [2, 3], [3, 4], [3, 5], [4, 5], [5, 7], [6, 4], [6, 7], [6, 8], [8, 7], [8, 9], [9, 2], [9, 7]]



In [6]:
for name in GOLDENS:
    print(f"=== {name}")
    for algo in ["CPC", "CMIIC"]:
        dag_first = dag_from_json(json.loads((RESULTS / first / f"{name}__{algo}.json").read_text()))
        dag_second = dag_from_json(json.loads((RESULTS / second / f"{name}__{algo}.json").read_text()))
        skeleton_first = skeleton_of(dag_first)
        skeleton_second = skeleton_of(dag_second)
        if skeleton_first == skeleton_second:
            print(f"  {algo:6} squelettes identiques")
        else:
            print(f"  {algo:6} squelettes differents")
            print(f"    seulement dans {first} : {sorted(skeleton_first.edges() - skeleton_second.edges())}")
            print(f"    seulement dans {second} : {sorted(skeleton_second.edges() - skeleton_first.edges())}")
    print()

=== collider
  CPC    squelettes identiques
  CMIIC  squelettes identiques

=== fork
  CPC    squelettes identiques
  CMIIC  squelettes identiques

=== chain
  CPC    squelettes identiques
  CMIIC  squelettes identiques

=== report_10v
  CPC    squelettes identiques
  CMIIC  squelettes identiques



In [7]:
!jupyter nbconvert --to html --no-input compare_agrum2_agrum3.ipynb

[NbConvertApp] Converting notebook compare_agrum2_agrum3.ipynb to html
[NbConvertApp] Writing 397016 bytes to compare_agrum2_agrum3.html
